In [4]:
import csv
import os
import socket
import threading
import time
from datetime import datetime

OUTPUT_DIR = os.path.expanduser('~/Desktop')

SAMPLE_HL7_ORDERS = [
    "MSH|^~\\&|HIS_ORD|UT_HOSP|LIS_SYS|LAB_01|20261114101500||ORM^O01|MSG001|P|2.5\n"
    "PID|1||UT-2026-001||WANG^MENGCHU||19960512|F\n"
    "ORC|NW|ORD2026111401|||||^^^20261114101500\n"
    "OBR|1|ORD2026111401|SPEC261114001|BIO-01^Liver Panel (ALT/AST)^LIS|||20261114102000|||||||SERUM",

    "MSH|^~\\&|HIS_ORD|UT_HOSP|LIS_SYS|LAB_01|20261114101800||ORM^O01|MSG002|P|2.5\n"
    "PID|1||UT-2026-002||TANAKA^TARO||19820820|M\n"
    "ORC|NW|ORD2026111402|||||^^^20261114101800\n"
    "OBR|1|ORD2026111402|SPEC261114002|HEM-01^CBC Hematology^LIS|||20261114102500|||||||EDTA-2K",

    "MSH|^~\\&|HIS_ORD|UT_HOSP|LIS_SYS|LAB_01|20261114102200||ORM^O01|MSG003|P|2.5\n"
    "PID|1||UT-2026-003||SUZUKI^HANAKO||19750310|F\n"
    "ORC|NW|ORD2026111403|||||^^^20261114102200\n"
    "OBR|1|ORD2026111403|ERR_99|BIO-02^Renal Panel^LIS|||20261114103000|||||||SERUM"
]

def parse_hl7_order(hl7_text):
    data = {}
    for line in hl7_text.strip().split('\n'):
        fields = line.split('|')
        seg = fields[0]
        if seg == 'PID':
            data['PATIENT_ID'] = fields[3]
            data['PATIENT_NAME'] = fields[5].replace('^', ' ')
            data['SEX'] = fields[8]
        elif seg == 'OBR':
            data['ORDER_ID'] = fields[2]
            data['SPECIMEN_ID'] = fields[3]
            test_info = fields[4].split('^')
            data['TEST_CODE'] = test_info[0]
            data['TEST_NAME'] = test_info[1] if len(test_info) > 1 else ''
            data['TUBE_TYPE'] = fields[15] if len(fields) > 15 else 'UNKNOWN'
    return data

def validate_and_build_zpl(order_data):
    spec_id = order_data.get('SPECIMEN_ID', '')
    if not spec_id.startswith('SPEC') or len(spec_id) != 13:
        return False, "INVALID_SPECIMEN_BARCODE_FORMAT", ""

    zpl_payload = (
        f"^XA\n"
        f"^FO30,25^A0N,26,26^FDHOSP: UT-CRESCENT | {order_data['TUBE_TYPE']}^FS\n"
        f"^FO30,60^A0N,24,24^FDPID: {order_data['PATIENT_ID']} ({order_data['SEX']})^FS\n"
        f"^FO30,92^A0N,22,22^FDTEST: {order_data['TEST_CODE']} - {order_data['TEST_NAME']}^FS\n"
        f"^FO30,125^BY2,3,65^BCN,65,Y,N,N^FD{spec_id}^FS\n"
        f"^XZ"
    )
    return True, "VALID_ZPL_READY", zpl_payload

def mock_barcode_printer_server(host, port, stop_event, received_logs):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as srv:
        srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        srv.bind((host, port))
        srv.listen(5)
        srv.settimeout(0.5)
        while not stop_event.is_set():
            try:
                conn, addr = srv.accept()
                with conn:
                    payload = conn.recv(4096).decode('utf-8', errors='replace')
                    if payload.startswith('^XA') and payload.endswith('^XZ'):
                        received_logs.append(payload)
                        conn.sendall(b"ACK:PRINT_JOB_OK")
                    else:
                        conn.sendall(b"NAK:INVALID_ZPL_SYNTAX")
            except socket.timeout:
                continue

def run_printer_connectivity_suite():
    HOST = '127.0.0.1'
    PORT = 19100
    stop_event = threading.Event()
    printer_memory = []

    server_thread = threading.Thread(
        target=mock_barcode_printer_server,
        args=(HOST, PORT, stop_event, printer_memory),
        daemon=True
    )
    server_thread.start()
    time.sleep(0.2)

    test_results = []
    valid_orders_for_svg = []

    for idx, raw_hl7 in enumerate(SAMPLE_HL7_ORDERS, start=1):
        order = parse_hl7_order(raw_hl7)
        is_valid, status_msg, zpl_cmd = validate_and_build_zpl(order)
        ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

        if not is_valid:
            test_results.append({
                'TEST_CASE': f"TC-{idx:02d}", 'TIMESTAMP': ts,
                'SPECIMEN_ID': order.get('SPECIMEN_ID'), 'PATIENT_ID': order.get('PATIENT_ID'),
                'VALIDATION': status_msg, 'TCP_SOCKET_STATUS': 'SKIPPED_BY_GUARD',
                'PRINTER_RESPONSE': 'N/A'
            })
            continue

        try:
            with socket.create_connection((HOST, PORT), timeout=2.0) as client:
                client.sendall(zpl_cmd.encode('utf-8'))
                ack = client.recv(1024).decode('utf-8')
                test_results.append({
                    'TEST_CASE': f"TC-{idx:02d}", 'TIMESTAMP': ts,
                    'SPECIMEN_ID': order['SPECIMEN_ID'], 'PATIENT_ID': order['PATIENT_ID'],
                    'VALIDATION': status_msg, 'TCP_SOCKET_STATUS': 'CONNECTED_PORT_9100',
                    'PRINTER_RESPONSE': ack
                })
                valid_orders_for_svg.append(order)
        except Exception as e:
            test_results.append({
                'TEST_CASE': f"TC-{idx:02d}", 'TIMESTAMP': ts,
                'SPECIMEN_ID': order['SPECIMEN_ID'], 'PATIENT_ID': order['PATIENT_ID'],
                'VALIDATION': status_msg, 'TCP_SOCKET_STATUS': f'ERROR: {e}',
                'PRINTER_RESPONSE': 'TIMEOUT'
            })

    stop_event.set()
    server_thread.join(timeout=1.0)

    csv_path = os.path.join(OUTPUT_DIR, 'printer_connection_test_log.csv')
    headers = ['TEST_CASE', 'TIMESTAMP', 'SPECIMEN_ID', 'PATIENT_ID', 'VALIDATION', 'TCP_SOCKET_STATUS', 'PRINTER_RESPONSE']
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        writer.writerows(test_results)

    if valid_orders_for_svg:
        sample = valid_orders_for_svg[0]
        bars = []
        pattern = [2, 1, 3, 1, 2, 2, 1, 3, 2, 1, 1, 2, 3, 1, 2, 1, 3, 2, 1, 2, 2, 1, 3, 1, 2, 1, 1, 3, 2, 1, 2, 2]
        x_pos = 45
        for i, w in enumerate(pattern):
            if i % 2 == 0:
                bars.append(f'<rect x="{x_pos}" y="118" width="{w*2.2:.1f}" height="52" fill="#111"/>')
            x_pos += w * 2.5

        svg_label = f'''<svg xmlns="http://w3.org" width="460" height="220" viewBox="0 0 460 220">
  <rect x="10" y="10" width="440" height="200" rx="12" fill="#FDFDFD" stroke="#333" stroke-width="2"/>
  <rect x="10" y="10" width="440" height="34" rx="10" fill="#1565C0"/>
  <text x="25" y="32" font-family="monospace, sans-serif" font-size="14" font-weight="bold" fill="#FFFFFF">CLINICAL LAB SPECIMEN LABEL (ZPL PREVIEW)</text>
  <text x="395" y="32" font-family="monospace, sans-serif" font-size="13" font-weight="bold" fill="#FFEB3B">{sample['TUBE_TYPE']}</text>
  <text x="28" y="68" font-family="monospace, sans-serif" font-size="14" font-weight="bold" fill="#111">SUBJ/PID: {sample['PATIENT_ID']} ({sample['SEX']})</text>
  <text x="28" y="92" font-family="monospace, sans-serif" font-size="13" fill="#333">ORDER: {sample['ORDER_ID']} | {sample['TEST_CODE']} ({sample['TEST_NAME']})</text>
  {"".join(bars)}
  <text x="220" y="188" font-family="monospace, sans-serif" font-size="14" font-weight="bold" text-anchor="middle" fill="#111">*{sample['SPECIMEN_ID']}*</text>
</svg>'''
        svg_path = os.path.join(OUTPUT_DIR, 'specimen_label_preview.svg')
        with open(svg_path, 'w', encoding='utf-8') as f:
            f.write(svg_label)

    print(f"Success! Files saved to your Desktop:\n- {csv_path}\n- {svg_path}")

run_printer_connectivity_suite()


Success! Files saved to your Desktop:
- /Users/mengchuwang/Desktop/printer_connection_test_log.csv
- /Users/mengchuwang/Desktop/specimen_label_preview.svg


In [6]:
import csv
import os
import socket
import threading
import time
from datetime import datetime
from IPython.display import SVG, display

OUTPUT_DIR = os.path.expanduser('~/Desktop')

SAMPLE_HL7_ORDERS = [
    "MSH|^~\\&|HIS_ORD|UT_HOSP|LIS_SYS|LAB_01|20261114101500||ORM^O01|MSG001|P|2.5\n"
    "PID|1||UT-2026-001||WANG^MENGCHU||19960512|F\n"
    "ORC|NW|ORD2026111401|||||^^^20261114101500\n"
    "OBR|1|ORD2026111401|SPEC261114001|BIO-01^Liver Panel (ALT/AST)^LIS|||20261114102000|||||||SERUM",

    "MSH|^~\\&|HIS_ORD|UT_HOSP|LIS_SYS|LAB_01|20261114101800||ORM^O01|MSG002|P|2.5\n"
    "PID|1||UT-2026-002||TANAKA^TARO||19820820|M\n"
    "ORC|NW|ORD2026111402|||||^^^20261114101800\n"
    "OBR|1|ORD2026111402|SPEC261114002|HEM-01^CBC Hematology^LIS|||20261114102500|||||||EDTA-2K",

    "MSH|^~\\&|HIS_ORD|UT_HOSP|LIS_SYS|LAB_01|20261114102200||ORM^O01|MSG003|P|2.5\n"
    "PID|1||UT-2026-003||SUZUKI^HANAKO||19750310|F\n"
    "ORC|NW|ORD2026111403|||||^^^20261114102200\n"
    "OBR|1|ORD2026111403|ERR_99|BIO-02^Renal Panel^LIS|||20261114103000|||||||SERUM"
]

def parse_hl7_order(hl7_text):
    data = {}
    for line in hl7_text.strip().split('\n'):
        fields = line.split('|')
        seg = fields[0]
        if seg == 'PID':
            data['PATIENT_ID'] = fields[3]
            data['PATIENT_NAME'] = fields[5].replace('^', ' ')
            data['SEX'] = fields[8]
        elif seg == 'OBR':
            data['ORDER_ID'] = fields[2]
            data['SPECIMEN_ID'] = fields[3]
            test_info = fields[4].split('^')
            data['TEST_CODE'] = test_info[0]
            data['TEST_NAME'] = test_info[1] if len(test_info) > 1 else ''
            data['TUBE_TYPE'] = fields[15] if len(fields) > 15 else 'UNKNOWN'
    return data

def validate_and_build_zpl(order_data):
    spec_id = order_data.get('SPECIMEN_ID', '')
    if not spec_id.startswith('SPEC') or len(spec_id) != 13:
        return False, "INVALID_SPECIMEN_BARCODE_FORMAT", ""

    zpl_payload = (
        f"^XA\n"
        f"^FO30,25^A0N,26,26^FDHOSP: UT-CRESCENT | {order_data['TUBE_TYPE']}^FS\n"
        f"^FO30,60^A0N,24,24^FDPID: {order_data['PATIENT_ID']} ({order_data['SEX']})^FS\n"
        f"^FO30,92^A0N,22,22^FDTEST: {order_data['TEST_CODE']} - {order_data['TEST_NAME']}^FS\n"
        f"^FO30,125^BY2,3,65^BCN,65,Y,N,N^FD{spec_id}^FS\n"
        f"^XZ"
    )
    return True, "VALID_ZPL_READY", zpl_payload

def mock_barcode_printer_server(host, port, stop_event, received_logs):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as srv:
        srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        srv.bind((host, port))
        srv.listen(5)
        srv.settimeout(0.5)
        while not stop_event.is_set():
            try:
                conn, addr = srv.accept()
                with conn:
                    payload = conn.recv(4096).decode('utf-8', errors='replace')
                    if payload.startswith('^XA') and payload.endswith('^XZ'):
                        received_logs.append(payload)
                        conn.sendall(b"ACK:PRINT_JOB_OK")
                    else:
                        conn.sendall(b"NAK:INVALID_ZPL_SYNTAX")
            except socket.timeout:
                continue

def run_all():
    HOST = '127.0.0.1'
    PORT = 19100
    stop_event = threading.Event()
    printer_memory = []

    server_thread = threading.Thread(
        target=mock_barcode_printer_server,
        args=(HOST, PORT, stop_event, printer_memory),
        daemon=True
    )
    server_thread.start()
    time.sleep(0.2)

    test_results = []
    valid_orders = []

    for idx, raw_hl7 in enumerate(SAMPLE_HL7_ORDERS, start=1):
        order = parse_hl7_order(raw_hl7)
        is_valid, status_msg, zpl_cmd = validate_and_build_zpl(order)
        ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

        if not is_valid:
            test_results.append({
                'TEST_CASE': f"TC-{idx:02d}", 'TIMESTAMP': ts,
                'SPECIMEN_ID': order.get('SPECIMEN_ID'), 'PATIENT_ID': order.get('PATIENT_ID'),
                'VALIDATION': status_msg, 'TCP_SOCKET_STATUS': 'SKIPPED_BY_GUARD',
                'PRINTER_RESPONSE': 'N/A'
            })
            continue

        try:
            with socket.create_connection((HOST, PORT), timeout=2.0) as client:
                client.sendall(zpl_cmd.encode('utf-8'))
                ack = client.recv(1024).decode('utf-8')
                test_results.append({
                    'TEST_CASE': f"TC-{idx:02d}", 'TIMESTAMP': ts,
                    'SPECIMEN_ID': order['SPECIMEN_ID'], 'PATIENT_ID': order['PATIENT_ID'],
                    'VALIDATION': status_msg, 'TCP_SOCKET_STATUS': 'CONNECTED_PORT_9100',
                    'PRINTER_RESPONSE': ack
                })
                valid_orders.append(order)
        except Exception as e:
            test_results.append({
                'TEST_CASE': f"TC-{idx:02d}", 'TIMESTAMP': ts,
                'SPECIMEN_ID': order['SPECIMEN_ID'], 'PATIENT_ID': order['PATIENT_ID'],
                'VALIDATION': status_msg, 'TCP_SOCKET_STATUS': f'ERROR: {e}',
                'PRINTER_RESPONSE': 'TIMEOUT'
            })

    stop_event.set()
    server_thread.join(timeout=1.0)

    # Save CSV Log
    csv_path = os.path.join(OUTPUT_DIR, 'printer_connection_test_log.csv')
    headers = ['TEST_CASE', 'TIMESTAMP', 'SPECIMEN_ID', 'PATIENT_ID', 'VALIDATION', 'TCP_SOCKET_STATUS', 'PRINTER_RESPONSE']
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        writer.writerows(test_results)

    # Save Chart 1: Specimen Barcode Label SVG
    sample = valid_orders[0]
    bars = []
    pattern = [2, 1, 3, 2, 1, 1, 2, 3, 1, 2, 2, 1, 3, 1, 2, 2, 1, 3, 2, 1, 2, 1, 3, 2, 1, 2, 1, 1, 2, 3]
    x_pos = 45
    for i, w in enumerate(pattern):
        if i % 2 == 0:
            bars.append(f'<rect x="{x_pos}" y="118" width="{w*2.2:.1f}" height="52" fill="#111"/>')
        x_pos += w * 2.5

    svg_label = f'''<svg xmlns="http://w3.org" width="460" height="220" viewBox="0 0 460 220">
  <rect x="10" y="10" width="440" height="200" rx="12" fill="#FDFDFD" stroke="#333" stroke-width="2"/>
  <rect x="10" y="10" width="440" height="34" rx="10" fill="#1565C0"/>
  <text x="25" y="32" font-family="monospace, sans-serif" font-size="14" font-weight="bold" fill="#FFFFFF">CLINICAL LAB SPECIMEN LABEL (ZPL PREVIEW)</text>
  <text x="395" y="32" font-family="monospace, sans-serif" font-size="13" font-weight="bold" fill="#FFEB3B">{sample['TUBE_TYPE']}</text>
  <text x="28" y="68" font-family="monospace, sans-serif" font-size="14" font-weight="bold" fill="#111">SUBJ/PID: {sample['PATIENT_ID']} ({sample['SEX']})</text>
  <text x="28" y="92" font-family="monospace, sans-serif" font-size="13" fill="#333">ORDER: {sample['ORDER_ID']} | {sample['TEST_CODE']} ({sample['TEST_NAME']})</text>
  {"".join(bars)}
  <text x="220" y="188" font-family="monospace, sans-serif" font-size="14" font-weight="bold" text-anchor="middle" fill="#111">*{sample['SPECIMEN_ID']}*</text>
</svg>'''
    label_svg_path = os.path.join(OUTPUT_DIR, 'specimen_label_preview.svg')
    with open(label_svg_path, 'w', encoding='utf-8') as f:
        f.write(svg_label)

    # Save Chart 2: Connectivity Test Summary SVG
    passed_cnt = sum(1 for r in test_results if r['PRINTER_RESPONSE'] == 'ACK:PRINT_JOB_OK')
    blocked_cnt = sum(1 for r in test_results if r['TCP_SOCKET_STATUS'] == 'SKIPPED_BY_GUARD')
    summary_svg = f'''<svg xmlns="http://w3.org" width="540" height="280" viewBox="0 0 540 280">
  <rect width="540" height="280" fill="#ffffff" rx="8" stroke="#e0e0e0"/>
  <text x="270" y="34" font-family="Arial, sans-serif" font-size="15" font-weight="bold" text-anchor="middle" fill="#222">HIS/LIS Barcode Printer Connectivity Test Summary</text>
  <line x1="70" y1="220" x2="470" y2="220" stroke="#888" stroke-width="1.5"/>
  <line x1="70" y1="60" x2="70" y2="220" stroke="#888" stroke-width="1.5"/>
  <rect x="130" y="{220 - passed_cnt*55}" width="80" height="{passed_cnt*55}" fill="#2E7D32" rx="4"/>
  <text x="170" y="{212 - passed_cnt*55}" font-family="Arial, sans-serif" font-size="14" font-weight="bold" text-anchor="middle" fill="#2E7D32">{passed_cnt} Passed</text>
  <text x="170" y="242" font-family="Arial, sans-serif" font-size="12" text-anchor="middle" fill="#333">TCP 9100 ACK OK</text>
  <rect x="320" y="{220 - blocked_cnt*55}" width="80" height="{blocked_cnt*55}" fill="#EF6C00" rx="4"/>
  <text x="360" y="{212 - blocked_cnt*55}" font-family="Arial, sans-serif" font-size="14" font-weight="bold" text-anchor="middle" fill="#EF6C00">{blocked_cnt} Blocked</text>
  <text x="360" y="242" font-family="Arial, sans-serif" font-size="12" text-anchor="middle" fill="#333">Invalid ID Guarded</text>
</svg>'''
    summary_svg_path = os.path.join(OUTPUT_DIR, 'printer_test_summary.svg')
    with open(summary_svg_path, 'w', encoding='utf-8') as f:
        f.write(summary_svg)

    print("Saved all files to Desktop!")
    display(SVG(filename=label_svg_path))
    display(SVG(filename=summary_svg_path))

run_all()


Saved all files to Desktop!


In [7]:
import os
from IPython.display import SVG, display

OUTPUT_DIR = os.path.expanduser('~/Desktop')

topology_svg = '''<svg xmlns="http://w3.org" width="900" height="440" viewBox="0 0 900 440">
  <defs>
    <marker id="arrow-blue" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 1 L 10 5 L 0 9 z" fill="#1565C0"/>
    </marker>
    <marker id="arrow-green" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 1 L 10 5 L 0 9 z" fill="#2E7D32"/>
    </marker>
    <marker id="arrow-orange" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 1 L 10 5 L 0 9 z" fill="#EF6C00"/>
    </marker>
  </defs>

  <!-- Background & Header -->
  <rect width="900" height="440" fill="#F8FAFC" rx="10" stroke="#CBD5E1" stroke-width="1.5"/>
  <rect x="0" y="0" width="900" height="46" fill="#0F172A" rx="10"/>
  <rect x="0" y="34" width="900" height="12" fill="#0F172A"/>
  <text x="450" y="29" font-family="Arial, sans-serif" font-size="16" font-weight="bold" text-anchor="middle" fill="#FFFFFF">
    Hospital EMR / HIS / LIS &amp; Clinical Lab Hardware Integration Simulation Topology
  </text>

  <!-- Zone 1: Hospital Core IT Zone -->
  <rect x="25" y="65" width="230" height="345" fill="#EFF6FF" rx="8" stroke="#93C5FD" stroke-width="1.5" stroke-dasharray="5,3"/>
  <text x="140" y="88" font-family="Arial, sans-serif" font-size="13" font-weight="bold" text-anchor="middle" fill="#1E40AF">Hospital Core IT Zone</text>

  <rect x="45" y="110" width="190" height="75" fill="#FFFFFF" rx="6" stroke="#2563EB" stroke-width="2"/>
  <text x="140" y="138" font-family="Arial, sans-serif" font-size="14" font-weight="bold" text-anchor="middle" fill="#1E3A8A">電子カルテ (EMR)</text>
  <text x="140" y="158" font-family="Arial, sans-serif" font-size="11" text-anchor="middle" fill="#475569">Doctor Order Entry</text>
  <text x="140" y="174" font-family="Arial, sans-serif" font-size="11" text-anchor="middle" fill="#475569">Result Viewer</text>

  <rect x="45" y="235" width="190" height="85" fill="#FFFFFF" rx="6" stroke="#2563EB" stroke-width="2"/>
  <text x="140" y="263" font-family="Arial, sans-serif" font-size="14" font-weight="bold" text-anchor="middle" fill="#1E3A8A">HIS Gateway Server</text>
  <text x="140" y="283" font-family="Arial, sans-serif" font-size="11" text-anchor="middle" fill="#475569">HL7 v2.5 ORM^O01 / ORU^R01</text>
  <text x="140" y="301" font-family="Arial, sans-serif" font-size="11" font-weight="bold" text-anchor="middle" fill="#0284C7">IP: 10.10.1.10 (Port 2575)</text>

  <line x1="140" y1="185" x2="140" y2="230" stroke="#1565C0" stroke-width="2" marker-end="url(#arrow-blue)" marker-start="url(#arrow-blue)"/>

  <!-- Zone 2: LIS Middleware & Validation Engine -->
  <rect x="315" y="65" width="265" height="345" fill="#F0FDF4" rx="8" stroke="#86EFAC" stroke-width="1.5" stroke-dasharray="5,3"/>
  <text x="447" y="88" font-family="Arial, sans-serif" font-size="13" font-weight="bold" text-anchor="middle" fill="#166534">LIS &amp; Integration Test Engine</text>

  <rect x="335" y="125" width="225" height="105" fill="#FFFFFF" rx="6" stroke="#16A34A" stroke-width="2"/>
  <text x="447" y="152" font-family="Arial, sans-serif" font-size="14" font-weight="bold" text-anchor="middle" fill="#14532D">LIS Core Server (HK-LAS)</text>
  <text x="447" y="172" font-family="Arial, sans-serif" font-size="11" text-anchor="middle" fill="#334155">Specimen ID &amp; Barcode Mapping</text>
  <text x="447" y="189" font-family="Arial, sans-serif" font-size="11" text-anchor="middle" fill="#334155">QC Range &amp; Delta Check</text>
  <text x="447" y="208" font-family="Arial, sans-serif" font-size="11" font-weight="bold" text-anchor="middle" fill="#15803D">IP: 10.10.2.20 | DB: PostgreSQL</text>

  <rect x="335" y="275" width="225" height="105" fill="#FEFCE8" rx="6" stroke="#CA8A04" stroke-width="2"/>
  <text x="447" y="302" font-family="Arial, sans-serif" font-size="13" font-weight="bold" text-anchor="middle" fill="#854D0E">Python Connectivity Simulator</text>
  <text x="447" y="322" font-family="Arial, sans-serif" font-size="11" text-anchor="middle" fill="#334155">1. Socket Loopback &amp; ACK/NAK Test</text>
  <text x="447" y="339" font-family="Arial, sans-serif" font-size="11" text-anchor="middle" fill="#334155">2. ZPL II Barcode Format Guard</text>
  <text x="447" y="356" font-family="Arial, sans-serif" font-size="11" text-anchor="middle" fill="#334155">3. ASTM / HL7 Packet Verification</text>

  <line x1="447" y1="230" x2="447" y2="270" stroke="#2E7D32" stroke-width="2" marker-end="url(#arrow-green)" marker-start="url(#arrow-green)"/>

  <!-- Connection HIS <-> LIS -->
  <line x1="235" y1="265" x2="330" y2="180" stroke="#1565C0" stroke-width="2.2" marker-end="url(#arrow-blue)"/>
  <text x="275" y="200" font-family="Arial, sans-serif" font-size="10" font-weight="bold" fill="#1E40AF" transform="rotate(-40 275,200)">HL7 Order</text>
  <line x1="330" y1="205" x2="235" y2="290" stroke="#2E7D32" stroke-width="2.2" marker-end="url(#arrow-green)"/>
  <text x="282" y="268" font-family="Arial, sans-serif" font-size="10" font-weight="bold" fill="#166534" transform="rotate(-40 282,268)">Lab Result</text>

  <!-- Zone 3: Clinical Lab Hardware Floor -->
  <rect x="640" y="65" width="235" height="345" fill="#FFF7ED" rx="8" stroke="#FDBA74" stroke-width="1.5" stroke-dasharray="5,3"/>
  <text x="757" y="88" font-family="Arial, sans-serif" font-size="13" font-weight="bold" text-anchor="middle" fill="#9A3412">Clinical Lab Hardware Floor</text>

  <!-- Hardware 1: Barcode Printer -->
  <rect x="660" y="110" width="195" height="80" fill="#FFFFFF" rx="6" stroke="#EA580C" stroke-width="2"/>
  <text x="757" y="135" font-family="Arial, sans-serif" font-size="13" font-weight="bold" text-anchor="middle" fill="#9A3412">Barcode Label Printer</text>
  <text x="757" y="153" font-family="Arial, sans-serif" font-size="11" text-anchor="middle" fill="#475569">Zebra / SATO (ZPL II / SBPL)</text>
  <text x="757" y="171" font-family="Arial, sans-serif" font-size="11" font-weight="bold" text-anchor="middle" fill="#C2410C">TCP Raw Socket Port 9100</text>

  <!-- Hardware 2: Automated Analyzer -->
  <rect x="660" y="212" width="195" height="80" fill="#FFFFFF" rx="6" stroke="#EA580C" stroke-width="2"/>
  <text x="757" y="237" font-family="Arial, sans-serif" font-size="13" font-weight="bold" text-anchor="middle" fill="#9A3412">Biochemistry Analyzer</text>
  <text x="757" y="255" font-family="Arial, sans-serif" font-size="11" text-anchor="middle" fill="#475569">ASTM E1381 / E1394 Protocol</text>
  <text x="757" y="273" font-family="Arial, sans-serif" font-size="11" font-weight="bold" text-anchor="middle" fill="#C2410C">RS-232C / TCP Serial Bridge</text>

  <!-- Hardware 3: Specimen Sorter -->
  <rect x="660" y="314" width="195" height="80" fill="#FFFFFF" rx="6" stroke="#EA580C" stroke-width="2"/>
  <text x="757" y="339" font-family="Arial, sans-serif" font-size="13" font-weight="bold" text-anchor="middle" fill="#9A3412">Specimen Tube Sorter</text>
  <text x="757" y="357" font-family="Arial, sans-serif" font-size="11" text-anchor="middle" fill="#475569">Optical Barcode Reader Check</text>
  <text x="757" y="375" font-family="Arial, sans-serif" font-size="11" font-weight="bold" text-anchor="middle" fill="#C2410C">Rack Routing &amp; Error Reject</text>

  <!-- Connections LIS <-> Hardware -->
  <line x1="560" y1="150" x2="655" y2="150" stroke="#EF6C00" stroke-width="2" marker-end="url(#arrow-orange)" marker-start="url(#arrow-orange)"/>
  <text x="608" y="142" font-family="Arial, sans-serif" font-size="10" font-weight="bold" text-anchor="middle" fill="#C2410C">ZPL / ACK</text>

  <line x1="560" y1="190" x2="655" y2="250" stroke="#EF6C00" stroke-width="2" marker-end="url(#arrow-orange)" marker-start="url(#arrow-orange)"/>
  <text x="612" y="210" font-family="Arial, sans-serif" font-size="10" font-weight="bold" text-anchor="middle" fill="#C2410C">ASTM</text>

  <line x1="560" y1="330" x2="655" y2="350" stroke="#EF6C00" stroke-width="2" marker-end="url(#arrow-orange)" marker-start="url(#arrow-orange)"/>
  <text x="608" y="328" font-family="Arial, sans-serif" font-size="10" font-weight="bold" text-anchor="middle" fill="#C2410C">Scan ID</text>
</svg>'''

svg_path = os.path.join(OUTPUT_DIR, 'his_lis_hardware_simulation_topology.svg')
with open(svg_path, 'w', encoding='utf-8') as f:
    f.write(topology_svg)

print(f"Saved floorplan topology diagram to: {svg_path}")
display(SVG(filename=svg_path))


Saved floorplan topology diagram to: /Users/mengchuwang/Desktop/his_lis_hardware_simulation_topology.svg
